In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

class Visualizer:
    @staticmethod
    def plot_combined_figure(
        l_syn_df, l_real_df, s_syn_df, s_real_df,
        l_synthetic_k_df, conv_m_df, T
    ):
        os.makedirs("split_figs", exist_ok=True)

        # ===================== preprocess =====================
        for df in [l_syn_df, l_real_df, s_syn_df, s_real_df,
                   l_synthetic_k_df, conv_m_df]:
            df.rename(columns={'algorithm': 'Algorithm'}, inplace=True)
            df['Algorithm'] = df['Algorithm'].replace({'EXP3_M': 'EXP3.M'})

        all_df = pd.concat([
            l_syn_df, l_real_df, s_syn_df, s_real_df,
            l_synthetic_k_df, conv_m_df
        ])
        algorithms = sorted(all_df['Algorithm'].dropna().unique().tolist())

        for df in [l_syn_df, l_real_df, s_syn_df, s_real_df,
                   l_synthetic_k_df, conv_m_df]:
            df['Algorithm'] = pd.Categorical(
                df['Algorithm'], categories=algorithms, ordered=True
            )

        # ===================== figure =====================
        sns.set_theme(style="white", font_scale=2.5)
        fig, axes = plt.subplots(2, 4, figsize=(32, 10), sharex=False)

        # ===================== helper function for line plots =====================
        def draw_line(ax, df, y_scale=1e3):
            sub = df[df["k"].isin([10])].copy()
            sub["t_scaled"] = sub["t"] / 1e4
            sub["regret_scaled"] = sub["regret"] / y_scale

            sns.lineplot(
                data=sub,
                x='t_scaled', y='regret_scaled',
                hue='Algorithm', hue_order=algorithms,
                estimator='mean', errorbar='sd',
                palette='tab10', ax=ax
            )

            ax.set_xlabel("Rounds ($\\times 10^4$)")
            ax.set_ylabel(f"Regret ($\\times 10^{{{int(abs(np.log10(y_scale)))}}}$)")
            ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
            ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3))
            ax.get_legend().remove()

        # ===================== draw 8 subplots =====================
        # Line plots
        draw_line(axes[0][0], l_syn_df, y_scale=1e3)
        draw_line(axes[0][1], s_syn_df, y_scale=1e2)   
        draw_line(axes[1][0], l_real_df, y_scale=1e3)
        draw_line(axes[1][1], s_real_df, y_scale=1e2) 


        # Bar plots
        bar_k = l_synthetic_k_df.copy()
        bar_k["regret_scaled"] = bar_k["regret"] / 1e3
        sns.barplot(
            data=bar_k, x='k', y='regret_scaled',
            hue='Algorithm', hue_order=algorithms,
            errorbar=None, palette='tab10', dodge=True,
            capsize=0.1, ax=axes[0][2]
        )
        axes[0][2].set_xlabel("$k$")
        axes[0][2].set_ylabel("Regret ($\\times 10^3$)")
        axes[0][2].get_legend().remove()

        sns.barplot(
            data=bar_k, x='k', y='cumulative_time',
            hue='Algorithm', hue_order=algorithms,
            errorbar=None, palette='tab10', dodge=True,
            capsize=0.1, ax=axes[0][3]
        )
        axes[0][3].set_xlabel("$k$")
        axes[0][3].set_ylabel("Runtime (sec)")
        axes[0][3].get_legend().remove()

        bar_m = conv_m_df.copy()
        bar_m["regret_scaled"] = bar_m["regret"] / 1e3
        sns.barplot(
            data=bar_m, x='m', y='regret_scaled',
            hue='Algorithm', hue_order=algorithms,
            errorbar=None, palette='tab10', dodge=True,
            capsize=0.1, ax=axes[1][2]
        )
        axes[1][2].set_xlabel("$m$")
        axes[1][2].set_ylabel("Regret ($\\times 10^3$)")
        axes[1][2].get_legend().remove()

        sns.barplot(
            data=conv_m_df, x='m', y='cumulative_time',
            hue='Algorithm', hue_order=algorithms,
            errorbar=None, palette='tab10', dodge=True,
            capsize=0.1, ax=axes[1][3]
        )
        axes[1][3].set_xlabel("$m$")
        axes[1][3].set_ylabel("Runtime (sec)")
        axes[1][3].get_legend().remove()

        fig.tight_layout(rect=[0.02, 0.03, 1, 0.95])
        fig.canvas.draw()

        # ===================== save each subplot separately =====================
        renderer = fig.canvas.get_renderer()
        subplot_labels = [f"({i})" for i in range(1, 9)]
        for idx, ax in enumerate(axes.flat):
            bbox = ax.get_tightbbox(renderer).transformed(fig.dpi_scale_trans.inverted())
            bbox = bbox.expanded(1.05, 1.05)  
            fig.savefig(
                f"split_figs/subplot_{idx+1:02d}{subplot_labels[idx]}.pdf",
                bbox_inches=bbox, pad_inches=0.02, dpi=600
            )
            fig.savefig(
                f"split_figs/subplot_{idx+1:02d}{subplot_labels[idx]}.png",
                bbox_inches=bbox, pad_inches=0.02, dpi=600
            )

        # ===================== export legends separately with spacers =====================
        handles_line, labels_line = axes[0][0].get_legend_handles_labels()
        handles_bar, labels_bar = axes[0][2].get_legend_handles_labels()

        fig_leg1 = plt.figure(figsize=(10, 2))  
        ax_leg1 = fig_leg1.add_axes([0,0,1,1])
        ax_leg1.axis('off')  
        fig_leg1.legend(
            handles_line, labels_line,
            loc="center",
            ncol=len(labels_line),
            frameon=False,
            prop={'size':24}
        )
        fig_leg1.savefig("split_figs/legend_line.pdf", bbox_inches="tight")
        fig_leg1.savefig("split_figs/legend_line.png", bbox_inches="tight", dpi=600)
        plt.close(fig_leg1)

        fig_leg2 = plt.figure(figsize=(10, 2))  
        ax_leg2 = fig_leg2.add_axes([0,0,1,1])
        ax_leg2.axis('off')
        fig_leg2.legend(
            handles_bar, labels_bar,
            loc="center",
            ncol=len(labels_bar),
            frameon=False,
            prop={'size':24}
        )
        fig_leg2.savefig("split_figs/legend_bar.pdf", bbox_inches="tight")
        fig_leg2.savefig("split_figs/legend_bar.png", bbox_inches="tight", dpi=600)
        plt.close(fig_leg2)


if __name__ == "__main__":
    T = 100000

    l_syn_df = pd.read_csv("./results/l_conv_syn.csv")
    l_syn_df = l_syn_df[l_syn_df['t'] <= T]

    s_syn_df = pd.read_csv("./results/s_conv_syn.csv")
    s_syn_df = s_syn_df[s_syn_df['t'] <= T]

    l_real_df = pd.read_csv("./results/l_conv_yelp.csv")
    l_real_df = l_real_df[l_real_df['t'] <= T]

    s_real_df = pd.read_csv("./results/s_conv_yelp.csv")
    s_real_df = s_real_df[s_real_df['t'] <= T]

    l_synthetic_k_df = pd.read_csv("./results/l_conv_synthetic_varying_k.csv")
    l_synthetic_k_df = l_synthetic_k_df[l_synthetic_k_df['k'].isin([5,10,15,20,25])]
    idx = l_synthetic_k_df.groupby(['sim_id','k','algorithm'])['t'].idxmax()
    l_synthetic_k_df = l_synthetic_k_df.loc[idx]

    l_conv_m_df = pd.read_csv("./results/l_conv_synthetic_varying_m.csv")
    l_conv_m_df = l_conv_m_df[l_conv_m_df['m'].isin([20,25,30,35,40])]
    l_conv_m_df = l_conv_m_df[l_conv_m_df['t'] <= T]
    idx = l_conv_m_df.groupby(['sim_id','m','algorithm'])['t'].idxmax()
    l_conv_m_df = l_conv_m_df.loc[idx]

    Visualizer.plot_combined_figure(
        l_syn_df, l_real_df, s_syn_df, s_real_df,
        l_synthetic_k_df, l_conv_m_df, T=T
    )
